
# 6. FEATURE ENGINEERING (ÖZNİTELİK MÜHENDİSLİĞİ)

**Proje:** Veri Madenciliği Teknikleri ile IoT/IIoT Ağlarında Siber Saldırı Tespiti ve Sınıflandırılması

**Veri Seti:** Edge-IIoTset

## 6.1. Eğitim ve Test Verilerinin Ayrılması (Train/Test Split)

**Amaç:** Makine öğrenmesi modellerinin eğitimi ve tarafsız değerlendirilmesi için çalışma veri setini eğitim ve test bölümlerine ayırmak.

**Yöntem:**

399.992 kayıt içeren temizlenmiş veri seti, %80 eğitim ve %20 test olmak üzere iki bölüme ayrılacaktır.

15 trafik sınıfının her iki bölümde de temsil edilmesi amacıyla Attack_type değişkenine göre Stratified Random Split yöntemi uygulanacaktır.

İşlemin tekrarlanabilir olması için random_state=42 kullanılacaktır.

**Beklenen Çıktı:**

- %80 eğitim veri seti
- %20 test veri seti
- Her iki veri setinde 63 sütunun korunması
- 15 trafik sınıfının eğitim ve test dağılımlarını gösteren tablo
- Ayrı CSV dosyaları

**Önemli Not:**

Eksik değer doldurma, kategorik kodlama, ölçeklendirme ve öznitelik seçimi işlemleri eğitim verileri üzerinde öğrenilecek ve test verilerine uygulanacaktır.

Azınlık sınıflarını koruyan örnekleme nedeniyle test kümesinin sınıf dağılımı orijinal ağ trafiğinin doğal dağılımını birebir temsil etmemektedir. Ayrıca benzer ağ oturumlarının iki kümeye dağılmasından kaynaklanabilecek veri sızıntısı riski sonraki model değerlendirmelerinde dikkate alınacaktır.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Proje klasörü
proje = Path(r"E:\Data mining\Network_Intrusion_Detection")

# Temizlenmiş veri seti
giris = proje / "Calisma_Verileri" / "EdgeIIoT_399992_Son_Temizlik.csv"

# Eğitim ve test dosyaları
egitim_dosyasi = proje / "Calisma_Verileri" / "EdgeIIoT_Train.csv"
test_dosyasi = proje / "Calisma_Verileri" / "EdgeIIoT_Test.csv"

# Yalnızca hedef değişkenlerini okuma
hedefler = pd.read_csv(
    giris,
    usecols=["Attack_type", "Attack_label"],
    dtype=str,
    na_filter=False
)

# Kayıt indekslerini ayırma
indeksler = np.arange(len(hedefler))

egitim_idx, test_idx = train_test_split(
    indeksler,
    test_size=0.20,
    random_state=42,
    stratify=hedefler["Attack_type"]
)

# Hangi kaydın hangi gruba ait olduğunu belirleme
egitim_maskesi = np.zeros(len(hedefler), dtype=bool)
egitim_maskesi[egitim_idx] = True

# Veri setini parçalar halinde CSV dosyalarına yazma
baslangic = 0
ilk_parca = True

for parca in pd.read_csv(
    giris,
    chunksize=20000,
    dtype=str,
    na_filter=False
):
    bitis = baslangic + len(parca)

    secim = egitim_maskesi[baslangic:bitis]

    parca.iloc[secim].to_csv(
        egitim_dosyasi,
        mode="w" if ilk_parca else "a",
        header=ilk_parca,
        index=False
    )

    parca.iloc[~secim].to_csv(
        test_dosyasi,
        mode="w" if ilk_parca else "a",
        header=ilk_parca,
        index=False
    )

    ilk_parca = False
    baslangic = bitis

# Sınıf dağılımlarını karşılaştırma
egitim_sinif = hedefler.iloc[egitim_idx]["Attack_type"].value_counts()
test_sinif = hedefler.iloc[test_idx]["Attack_type"].value_counts()

dagilim = pd.DataFrame({
    "Eğitim": egitim_sinif,
    "Test": test_sinif
}).fillna(0).astype(int)

print("TRAIN / TEST SPLIT SONUÇLARI")
print("-" * 40)
print("Toplam kayıt:", len(hedefler))
print("Eğitim kayıtları:", len(egitim_idx))
print("Test kayıtları:", len(test_idx))
print("Sınıf sayısı:", len(dagilim))

display(dagilim)

# Doğrulama
assert len(hedefler) == 399992
assert len(dagilim) == 15
assert len(egitim_idx) + len(test_idx) == len(hedefler)
assert not np.any(egitim_maskesi[test_idx])

print("\nEğitim dosyası:", egitim_dosyasi)
print("Test dosyası:", test_dosyasi)
print("\nTrain/Test ayrımı tamamlandı.")


TRAIN / TEST SPLIT SONUÇLARI
----------------------------------------
Toplam kayıt: 399992
Eğitim kayıtları: 319993
Test kayıtları: 79999
Sınıf sayısı: 15


,Eğitim,Test
Attack_type,,
Backdoor,379,95
DDoS_HTTP,3494,874
DDoS_ICMP,44465,11116
DDoS_TCP,30822,7705
DDoS_UDP,48847,12212
MITM,240,60
Normal,171026,42757
OS_Fingerprinting,240,60
Password,16072,4018



Eğitim dosyası: E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri\EdgeIIoT_Train.csv
Test dosyası: E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri\EdgeIIoT_Test.csv

Train/Test ayrımı tamamlandı.



### 6.1. Eğitim ve Test Verilerinin Ayrılması — Sonuç ve Yorum

Edge-IIoTset çalışma veri setindeki 399.992 kayıt, tabakalı rastgele ayırma (Stratified Random Split) yöntemiyle eğitim ve test kümelerine ayrılmıştır.

**Elde Edilen Sonuçlar:**

- Toplam kayıt: 399.992
- Eğitim verisi: 319.993 (%80)
- Test verisi: 79.999 (%20)
- Trafik sınıfı sayısı: 15

**Değerlendirme:**

Attack_type değişkenine göre yapılan tabakalı ayırma sayesinde 15 trafik sınıfı her iki veri kümesinde de temsil edilmiştir.

Örneğin MITM, Ransomware ve OS_Fingerprinting sınıflarının her birinde 240 eğitim ve 60 test kaydı bulunmaktadır.

**Sonuç:**

Eğitim ve test veri kümeleri oluşturularak ayrı CSV dosyalarına kaydedilmiştir.

Eksik değer doldurma, kodlama ve ölçeklendirme işlemleri yalnızca eğitim verisinden öğrenilecektir.

Bununla birlikte örnekleme yöntemi ve benzer ağ kayıtlarının iki kümeye dağılma olasılığı, model performansının yorumlanmasında dikkate alınacaktır.



## 6.2. Eksik Değerlerin Doldurulması

**Amaç:** Veri temizleme aşamasında oluşturulan eksik değerleri makine öğrenmesi modellerine uygun şekilde düzenlemek.

**Yöntem:**

Yedi sayısal öznitelikteki eksik değerler, eğitim kümesinden hesaplanan medyan değerlerle doldurulacaktır.

Eksik değer bulunan hücrelerin bilgisi, her öznitelik için oluşturulan 0/1 göstergeleriyle korunacaktır.

Test verilerinde de eğitim kümesinden hesaplanan medyanlar kullanılacaktır.

**Beklenen Çıktı:**

- Eksik değer sayılarının karşılaştırılması
- Eğitim verisinden hesaplanan medyan değerler
- Eksik değerleri doldurulmuş eğitim ve test CSV dosyaları

**Not:** Kaynak CSV dosyaları korunacak, hedef değişkenler değiştirilmeyecektir.


In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

# Proje dosyaları
klasor = Path(
    r"E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri"
)

sutunlar = [
    "tcp.srcport", "tcp.dstport", "tcp.len",
    "udp.port", "arp.hw.size",
    "dns.qry.name.len", "tcp.connection.fin"
]

# Medyanları yalnızca eğitim verisinden hesaplama
egitim = pd.read_csv(
    klasor / "EdgeIIoT_Train.csv",
    usecols=sutunlar,
    dtype=str,
    na_filter=False
)

medyanlar = {}

for sutun in sutunlar:
    sayisal = pd.to_numeric(
        egitim[sutun], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    medyanlar[sutun] = sayisal.median()
    assert pd.notna(medyanlar[sutun])

print("EĞİTİM VERİSİNDEN HESAPLANAN MEDYANLAR:")
display(pd.Series(medyanlar, name="Medyan").to_frame())

# Eğitim ve test dosyalarını ayrı ayrı işleme
ozet = []

for grup in ["Train", "Test"]:

    giris = klasor / f"EdgeIIoT_{grup}.csv"
    cikis = klasor / f"EdgeIIoT_{grup}_Doldurulmus.csv"

    ilk_parca = True
    toplam = 0
    eksik_sayisi = 0

    for parca in pd.read_csv(
        giris,
        chunksize=20000,
        dtype=str,
        na_filter=False
    ):
        toplam += len(parca)

        for sutun in sutunlar:
            sayisal = pd.to_numeric(
                parca[sutun], errors="coerce"
            ).replace([np.inf, -np.inf], np.nan)

            eksik = sayisal.isna()
            eksik_sayisi += int(eksik.sum())

            # Eksik değer göstergesi
            parca[sutun + "_eksik"] = eksik.astype(int)

            # Eğitim medyanıyla doldurma
            parca[sutun] = sayisal.fillna(medyanlar[sutun])

        parca.to_csv(
            cikis,
            mode="w" if ilk_parca else "a",
            header=ilk_parca,
            index=False
        )

        ilk_parca = False

    ozet.append({
        "Veri Kümesi": grup,
        "Kayıt": toplam,
        "Doldurulan Hücre": eksik_sayisi
    })

# Sonuçlar
print("\nEKSİK DEĞER DOLDURMA SONUÇLARI:")
ozet_tablosu = pd.DataFrame(ozet)
display(ozet_tablosu)

assert ozet_tablosu["Kayıt"].sum() == 399992
assert ozet_tablosu["Doldurulan Hücre"].sum() == 3283

print("\nEksik değer doldurma işlemi tamamlandı.")


EĞİTİM VERİSİNDEN HESAPLANAN MEDYANLAR:


,Medyan
tcp.srcport,4321.0
tcp.dstport,1883.0
tcp.len,0.0
udp.port,0.0
arp.hw.size,0.0
dns.qry.name.len,0.0
tcp.connection.fin,0.0



EKSİK DEĞER DOLDURMA SONUÇLARI:


,Veri Kümesi,Kayıt,Doldurulan Hücre
0,Train,319993,2643
1,Test,79999,640



Eksik değer doldurma işlemi tamamlandı.



### 6.2. Eksik Değerlerin Doldurulması — Sonuç ve Yorum

Veri temizleme aşamasında oluşturulan 3.283 eksik hücre, eğitim verilerinden hesaplanan medyan değerlerle doldurulmuştur.

**Elde Edilen Sonuçlar:**

| Veri Kümesi | Kayıt | Doldurulan Hücre |
|---|---:|---:|
| Eğitim | 319.993 | 2.643 |
| Test | 79.999 | 640 |
| Toplam | 399.992 | 3.283 |

tcp.srcport için medyan 4.321, tcp.dstport için 1.883 olarak hesaplanmıştır. Diğer beş özniteliğin medyanı sıfırdır.

Eksik değerlerin önceden bulunduğu konumları belirtmek amacıyla yedi ek gösterge özniteliği oluşturulmuştur.

**Değerlendirme:**

Medyanlar yalnızca eğitim kümesinden hesaplanmış ve aynı değerler test kümesine uygulanmıştır. Böylece eksik değer doldurma işleminde test verilerinden bilgi alınmamıştır.

Bazı özniteliklerde medyanın sıfır olması, doldurulan hücrelerin protokole özgü gerçek sıfır değerlerle karışmasına neden olabilir. Oluşturulan eksiklik göstergeleri bu ayrımın korunmasını amaçlamaktadır.

**Sonuç:**

Belirlenen yedi öznitelikte eksik değer doldurma işlemi tamamlanmış, eğitim ve test verileri ayrı CSV dosyalarına kaydedilmiştir.

Diğer özniteliklerin veri türleri ve modelleme uygunluğu sonraki aşamalarda değerlendirilecektir.



## 6.3. Kategorik Verilerin Kodlanması (Encoding)

**Amaç:** Kategorik ağ trafiği özniteliklerini makine öğrenmesi modellerinin kullanabileceği sayısal biçime dönüştürmek.

**Yöntem:** http.request.method, http.request.version ve mqtt.protoname özniteliklerine One-Hot Encoding uygulanacaktır.

Kodlayıcı yalnızca eğitim verilerinde öğrenilecek ve test verilerine aynı kurallar uygulanacaktır.

Çok sayıda farklı değer içeren alanlarda oluşacak sütun sayısı sınırlandırılacaktır.

**Beklenen Çıktı:** Kategorik özniteliklerin sayısal göstergelere dönüştürülmesi ve kodlanmış eğitim/test CSV dosyalarının oluşturulması.

**Not:** Bu aşamada diğer metinsel öznitelikler değiştirilmemektedir. Nihai model özniteliklerinin belirlenmesi sonraki aşamada gerçekleştirilecektir.


In [3]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder

klasor = Path(
    r"E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri"
)

# Kodlanacak kategorik öznitelikler
kategorik = [
    "http.request.method",
    "http.request.version",
    "mqtt.protoname"
]

# Kodlayıcıyı yalnızca eğitim verisinden öğrenme
egitim_kategorik = pd.read_csv(
    klasor / "EdgeIIoT_Train_Doldurulmus.csv",
    usecols=kategorik,
    dtype=str,
    na_filter=False
)

kodlayici = OneHotEncoder(
    handle_unknown="infrequent_if_exist",
    max_categories=10,
    sparse_output=False,
    dtype=np.uint8
)

kodlayici.fit(egitim_kategorik[kategorik])

yeni_sutunlar = kodlayici.get_feature_names_out(kategorik)

print("Kodlanan kategorik sütun:", len(kategorik))
print("Oluşturulan gösterge sütunu:", len(yeni_sutunlar))

# Eğitim ve test dosyalarını aynı kodlayıcıyla dönüştürme
for grup in ["Train", "Test"]:

    giris = klasor / f"EdgeIIoT_{grup}_Doldurulmus.csv"
    cikis = klasor / f"EdgeIIoT_{grup}_Kodlanmis.csv"

    ilk_parca = True
    toplam = 0

    for parca in pd.read_csv(
        giris,
        dtype=str,
        na_filter=False,
        chunksize=20000
    ):

        toplam += len(parca)

        # Eğitimden öğrenilen kodlayıcıyı uygulama
        kodlar = kodlayici.transform(parca[kategorik])

        for i, sutun in enumerate(yeni_sutunlar):
            parca[sutun] = kodlar[:, i]

        # Kodlanan orijinal metin sütunlarını kaldırma
        parca = parca.drop(columns=kategorik)

        parca.to_csv(
            cikis,
            mode="w" if ilk_parca else "a",
            header=ilk_parca,
            index=False
        )

        ilk_parca = False

    print(f"{grup}: {toplam} kayıt kodlandı.")
    print("Dosya:", cikis.name)

    beklenen = 319993 if grup == "Train" else 79999
    assert toplam == beklenen

print("\nKategorik kodlama tamamlandı.")


Kodlanan kategorik sütun: 3
Oluşturulan gösterge sütunu: 17
Train: 319993 kayıt kodlandı.
Dosya: EdgeIIoT_Train_Kodlanmis.csv
Test: 79999 kayıt kodlandı.
Dosya: EdgeIIoT_Test_Kodlanmis.csv

Kategorik kodlama tamamlandı.



### 6.3. Kategorik Verilerin Kodlanması — Sonuç ve Yorum

Edge-IIoTset çalışma veri setinde seçilen üç kategorik özniteliğe One-Hot Encoding uygulanmıştır.

**Elde Edilen Sonuçlar:**

- Kodlanan kategorik öznitelik: 3
- Oluşturulan gösterge özniteliği: 17
- Eğitim kayıtları: 319.993
- Test kayıtları: 79.999

**Değerlendirme:**

http.request.method, http.request.version ve mqtt.protoname öznitelikleri sayısal gösterge sütunlarına dönüştürülmüştür.

Kodlayıcı yalnızca eğitim verisi üzerinde öğrenilmiş, aynı kodlama kuralları test verisine uygulanmıştır.

Böylece test verilerinden bilgi öğrenilmeden tutarlı bir kategorik dönüşüm sağlanmıştır.

**Sonuç:**

Seçilen üç kategorik özniteliğin kodlanması tamamlanmış ve eğitim/test dosyaları ayrı ayrı kaydedilmiştir.

Diğer metinsel öznitelikler henüz dönüştürülmemiştir. Bu nedenle elde edilen dosyalar model eğitimine doğrudan hazır kabul edilmemektedir.



## 6.4. Sayısal Özniteliklerin Ölçeklendirilmesi ve Kontrolü

**Amaç:** Eğitim ve test verilerindeki sayısal öznitelikleri makine öğrenmesi modelleri için uygun ölçeğe getirmek.

**Yöntem:** Eğitim verisinde tamamen sayısal olan öznitelikler belirlenecektir. Sayısal olmayan öznitelikler bu ilk model veri sürümüne dahil edilmeyecektir.

Sürekli sayısal özniteliklere StandardScaler uygulanacaktır. One-Hot ve eksik değer göstergeleri ölçeklendirilmeyecektir.

Ölçeklendirme parametreleri yalnızca eğitim kümesinden öğrenilecek ve test kümesine uygulanacaktır.

**Beklenen Çıktı:** Sayısal özniteliklerden oluşan eğitim ve test dosyaları, öznitelik sayıları ve doğrulama sonuçları.

**Not:** Bu işlem nihai öznitelik seçimi değildir. Dışarıda bırakılan metinsel alanlar kaynak dosyalarda korunmaktadır.


In [4]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib

klasor = Path(r"E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri")

train = klasor / "EdgeIIoT_Train_Kodlanmis.csv"
test = klasor / "EdgeIIoT_Test_Kodlanmis.csv"

hedefler = ["Attack_label", "Attack_type"]

sutunlar = pd.read_csv(train, nrows=0).columns.tolist()
oznitelikler = [s for s in sutunlar if s not in hedefler]

# Sayısal öznitelikleri yalnızca eğitim verisinden belirleme
sayisal_adaylar = set(oznitelikler)

for parca in pd.read_csv(
    train, chunksize=20000, dtype=str, na_filter=False
):
    for sutun in list(sayisal_adaylar):
        degerler = pd.to_numeric(
            parca[sutun], errors="coerce"
        )
        if not np.isfinite(degerler).all():
            sayisal_adaylar.remove(sutun)

sayisal_sutunlar = [
    s for s in oznitelikler if s in sayisal_adaylar
]

metinsel_sutunlar = [
    s for s in oznitelikler if s not in sayisal_adaylar
]

# One-Hot ve eksiklik göstergelerini ölçek dışında tutma
gosterge_on_ekleri = (
    "http.request.method_",
    "http.request.version_",
    "mqtt.protoname_"
)

olcek_sutunlari = [
    s for s in sayisal_sutunlar
    if not s.endswith("_eksik")
    and not s.startswith(gosterge_on_ekleri)
]

# Ölçekleyiciyi yalnızca eğitim verisine uydurma
olcekleyici = StandardScaler()

for parca in pd.read_csv(
    train,
    usecols=olcek_sutunlari,
    chunksize=20000
):
    olcekleyici.partial_fit(parca[olcek_sutunlari])

# Eğitim ve test verilerini aynı kurallarla dönüştürme
for ad, giris in [("Train", train), ("Test", test)]:

    cikis = klasor / f"EdgeIIoT_{ad}_Sayisal_Model.csv"
    ilk = True
    toplam = 0

    for parca in pd.read_csv(
        giris, chunksize=20000, dtype=str, na_filter=False
    ):
        X = parca[sayisal_sutunlar].apply(
            pd.to_numeric, errors="coerce"
        )

        assert np.isfinite(X.to_numpy(dtype=float)).all(), (
            f"{ad}: Beklenmeyen sayısal uyumsuzluk!"
        )

        X[olcek_sutunlari] = olcekleyici.transform(
            X[olcek_sutunlari]
        )

        X["Attack_label"] = parca["Attack_label"].values
        X["Attack_type"] = parca["Attack_type"].values

        X.to_csv(
            cikis,
            mode="w" if ilk else "a",
            header=ilk,
            index=False
        )

        toplam += len(X)
        ilk = False

    print(f"{ad}: {toplam} kayıt kaydedildi.")
    assert toplam == (319993 if ad == "Train" else 79999)

# Ölçekleme parametrelerini saklama
joblib.dump(
    {
        "scaler": olcekleyici,
        "sayisal_sutunlar": sayisal_sutunlar,
        "olcek_sutunlari": olcek_sutunlari,
        "metinsel_sutunlar": metinsel_sutunlar
    },
    klasor / "06_On_Isleme_Bilgileri.joblib"
)

print("\n6.4 - SON KONTROL")
print("Sayısal giriş özniteliği:", len(sayisal_sutunlar))
print("Ölçeklendirilen:", len(olcek_sutunlari))
print("Bu sürüme alınmayan metinsel:", len(metinsel_sutunlar))
print("\nİşlem tamamlandı.")


Train: 319993 kayıt kaydedildi.
Test: 79999 kayıt kaydedildi.

6.4 - SON KONTROL
Sayısal giriş özniteliği: 60
Ölçeklendirilen: 36
Bu sürüme alınmayan metinsel: 22

İşlem tamamlandı.



### 6.4. Sayısal Özniteliklerin Ölçeklendirilmesi — Sonuç ve Yorum

Edge-IIoTset çalışma veri setinde sayısal öznitelikler makine öğrenmesi modellerine uygun biçimde hazırlanmıştır.

**Elde Edilen Sonuçlar:**

- Eğitim kayıt sayısı: 319.993
- Test kayıt sayısı: 79.999
- Sayısal giriş özniteliği: 60
- Ölçeklendirilen öznitelik: 36
- Bu model sürümüne dahil edilmeyen metinsel öznitelik: 22

**Değerlendirme:**

Seçilen sayısal özniteliklere StandardScaler yöntemi uygulanmıştır.

Ölçeklendirme parametreleri yalnızca eğitim verisinden öğrenilmiş ve test verilerine aynı parametreler uygulanmıştır.

One-Hot Encoding ve eksik değer göstergeleri ölçeklendirme dışında tutulmuştur.

Sayısal olmayan 22 öznitelik, bu ilk model sürümüne dahil edilmemiştir. Ancak söz konusu öznitelikler kaynak dosyalarda korunmaktadır.

**Sonuç:**

Sayısal modelleme için 60 giriş özniteliği içeren eğitim ve test veri kümeleri hazırlanmıştır.

Feature Engineering aşaması tamamlanmıştır.

Sonraki aşamada özniteliklerin sınıflandırma açısından katkıları incelenerek Feature Selection uygulanacaktır.



## 6.5. Yeni Özniteliklerin Oluşturulması

**Amaç:** Mevcut ağ trafiği verilerinden sınıflandırma için yararlı olabilecek yeni öznitelikler üretmek.

**Yöntem:** TCP uzunluğu, kaynak ve hedef port bilgileri ile HTTP istek yöntemlerinden dört yeni öznitelik türetilecektir.

- tcp_uzunluk_log: TCP uzunluğunun log(1+x) dönüşümü.
- tcp_port_cifti_var: Kaynak ve hedef TCP portlarının birlikte mevcut olma göstergesi.
- tcp_hedef_bilinen_port: 1–1023 aralığındaki hedef TCP portlarının göstergesi.
- http_istegi_var: Tanınan HTTP istek yöntemlerinin varlık göstergesi.

Eksik değer göstergeleri dikkate alınacak, doldurulmuş port değerleri gerçek gözlem gibi değerlendirilmeyecektir.

Aynı kurallar eğitim ve test verilerine uygulanacaktır.

**Beklenen Çıktı:** Mevcut 60 giriş özniteliğine dört yeni öznitelik eklenerek 64 giriş özniteliği içeren eğitim ve test dosyalarının oluşturulması.

**Not:** Yeni özniteliklerin gerçekten yararlı olup olmadığı sonraki Feature Selection ve model değerlendirme aşamalarında araştırılacaktır.


In [5]:

import pandas as pd
import numpy as np
from pathlib import Path
from itertools import zip_longest

# Proje klasörü
klasor = Path(
    r"E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri"
)

# Yeni oluşturulacak öznitelikler
yeni_oznitelikler = [
    "tcp_uzunluk_log",
    "tcp_port_cifti_var",
    "tcp_hedef_bilinen_port",
    "http_istegi_var"
]

# Gerekli ham ve eksiklik bilgilerini içeren sütunlar
kaynak_sutunlar = [
    "Attack_label", "Attack_type",
    "tcp.len", "tcp.srcport", "tcp.dstport",
    "tcp.srcport_eksik", "tcp.dstport_eksik",
    "http.request.method"
]

http_yontemleri = {
    "GET", "POST", "PUT", "DELETE", "HEAD",
    "OPTIONS", "PATCH", "TRACE", "CONNECT"
}

for grup, beklenen in [("Train", 319993), ("Test", 79999)]:

    # Ölçeklendirilmiş model dosyası
    model_dosyasi = (
        klasor / f"EdgeIIoT_{grup}_Sayisal_Model.csv"
    )

    # Ölçeklendirme öncesindeki veri
    kaynak_dosyasi = (
        klasor / f"EdgeIIoT_{grup}_Doldurulmus.csv"
    )

    cikis = klasor / f"EdgeIIoT_{grup}_Muhendislik.csv"

    model_okuyucu = pd.read_csv(
        model_dosyasi,
        dtype=str,
        na_filter=False,
        chunksize=20000
    )

    kaynak_okuyucu = pd.read_csv(
        kaynak_dosyasi,
        usecols=kaynak_sutunlar,
        dtype=str,
        na_filter=False,
        chunksize=20000
    )

    toplam = 0
    ilk = True
    ozellik_toplamlari = {
        ad: 0 for ad in yeni_oznitelikler[1:]
    }

    for model, kaynak in zip_longest(
        model_okuyucu, kaynak_okuyucu
    ):

        assert model is not None and kaynak is not None
        assert len(model) == len(kaynak)

        # Satır sırasının ve hedeflerin korunduğunu kontrol etme
        for hedef in ["Attack_label", "Attack_type"]:
            assert np.array_equal(
                model[hedef].to_numpy(),
                kaynak[hedef].to_numpy()
            )

        tcp_len = pd.to_numeric(
            kaynak["tcp.len"], errors="coerce"
        )
        src = pd.to_numeric(
            kaynak["tcp.srcport"], errors="coerce"
        )
        dst = pd.to_numeric(
            kaynak["tcp.dstport"], errors="coerce"
        )

        src_eksik = kaynak["tcp.srcport_eksik"].eq("1")
        dst_eksik = kaynak["tcp.dstport_eksik"].eq("1")

        assert tcp_len.notna().all()
        assert (tcp_len >= 0).all()

        # 1. TCP uzunluğunun log dönüşümü
        model["tcp_uzunluk_log"] = np.log1p(tcp_len)

        # 2. İki TCP portunun da bulunması
        model["tcp_port_cifti_var"] = (
            (src > 0) & (dst > 0)
            & ~src_eksik & ~dst_eksik
        ).astype(int)

        # 3. Bilinen hedef port aralığı
        model["tcp_hedef_bilinen_port"] = (
            dst.between(1, 1023)
            & ~dst_eksik
        ).astype(int)

        # 4. HTTP isteği varlığı
        model["http_istegi_var"] = (
            kaynak["http.request.method"]
            .str.strip()
            .str.upper()
            .isin(http_yontemleri)
        ).astype(int)

        for ad in ozellik_toplamlari:
            ozellik_toplamlari[ad] += int(model[ad].sum())

        model.to_csv(
            cikis,
            mode="w" if ilk else "a",
            header=ilk,
            index=False
        )

        toplam += len(model)
        ilk = False

    assert toplam == beklenen

    print(f"\n{grup.upper()} SONUÇLARI")
    print("Kayıt sayısı:", toplam)
    print("Yeni öznitelik sayısı:", 4)

    for ad, sayi in ozellik_toplamlari.items():
        print(f"{ad}: {sayi} pozitif kayıt")

    print("Kaydedilen dosya:", cikis.name)

# Nihai sütun sayısı kontrolü
for grup in ["Train", "Test"]:
    dosya = klasor / f"EdgeIIoT_{grup}_Muhendislik.csv"
    sutunlar = pd.read_csv(dosya, nrows=0).columns

    assert len(sutunlar) == 66
    assert all(ad in sutunlar for ad in yeni_oznitelikler)

print("\nİşlem tamamlandı!")
print("64 giriş özniteliği + 2 hedef sütunu = 66 sütun")



TRAIN SONUÇLARI
Kayıt sayısı: 319993
Yeni öznitelik sayısı: 4
tcp_port_cifti_var: 223491 pozitif kayıt
tcp_hedef_bilinen_port: 30434 pozitif kayıt
http_istegi_var: 2136 pozitif kayıt
Kaydedilen dosya: EdgeIIoT_Train_Muhendislik.csv

TEST SONUÇLARI
Kayıt sayısı: 79999
Yeni öznitelik sayısı: 4
tcp_port_cifti_var: 55870 pozitif kayıt
tcp_hedef_bilinen_port: 7431 pozitif kayıt
http_istegi_var: 503 pozitif kayıt
Kaydedilen dosya: EdgeIIoT_Test_Muhendislik.csv

İşlem tamamlandı!
64 giriş özniteliği + 2 hedef sütunu = 66 sütun



### 6.5. Yeni Özniteliklerin Oluşturulması — Sonuç ve Yorum

Mevcut ağ trafiği verilerinden dört yeni öznitelik oluşturulmuştur.

**Oluşturulan Öznitelikler:**

- `tcp_uzunluk_log`: TCP uzunluğunun logaritmik dönüşümü.
- `tcp_port_cifti_var`: Kaynak ve hedef TCP portlarının birlikte bulunma göstergesi.
- `tcp_hedef_bilinen_port`: Hedef portun 1–1023 aralığında bulunma göstergesi.
- `http_istegi_var`: Tanınan HTTP istek yöntemlerinin bulunma göstergesi.

**Elde Edilen Sonuçlar:**

| Gösterge | Eğitim | Test |
|---|---:|---:|
| Kayıt sayısı | 319.993 | 79.999 |
| TCP port çifti bulunan kayıt | 223.491 | 55.870 |
| Bilinen hedef port aralığı göstergesi | 30.434 | 7.431 |
| HTTP isteği göstergesi | 2.136 | 503 |

**Değerlendirme:**

Yeni öznitelikler, mevcut trafik bilgilerinden önceden belirlenen kurallar kullanılarak türetilmiştir.

Eğitim ve test kümelerine aynı dönüşümler uygulanmış, hedef değişkenler korunmuştur.

Böylece giriş özniteliği sayısı 60'tan 64'e yükselmiştir.

**Sonuç:**

Feature Engineering aşaması tamamlanmıştır. Oluşturulan veri setleri 64 giriş özniteliği ve 2 hedef değişken olmak üzere toplam 66 sütun içermektedir.

Yeni özniteliklerin sınıflandırma performansına katkısı, sonraki Feature Selection ve model değerlendirme aşamalarında araştırılacaktır.
